In [0]:
Landing_layer_path="/Volumes/pricing_data/deltalake/landing/"
Folder_Name=dbutils.widgets.get("Folder_Name")
Landing_layer_full_path=Landing_layer_path+Folder_Name+'/'

quarntine_path="/Volumes/pricing_data/deltalake/quarntine"
quarntine_full_path=quarntine_path+Folder_Name+'/'

bronze_path="/Volumes/pricing_data/deltalake/bronze"
bronze_full_path=bronze_path+'/'+Folder_Name+'/'
bronze_catalog_path=f"pricing_data.bronze.{Folder_Name}"

In [0]:
if len(dbutils.fs.ls(Landing_layer_full_path))==0:
    dbutils.notebook.exit("The Landing Folder Empty!!")

In [0]:
ingested_folder_data=spark.sql(f"""select file_name from pricing_data.process_run_logs.pipeline_filevalidation_logs where folder_name='{Folder_Name}' and file_status='Pass' and process_name='pricing_data_ingestion_bronze'""").collect()

processed_folder_data=[Folder_name.file_name for Folder_name in ingested_folder_data]

landing_layer_data=[Folder_name.name.replace('/', '') for Folder_name in dbutils.fs.ls(Landing_layer_full_path)]

un_processed_folders=list(set(landing_layer_data)-set(processed_folder_data))

if len(un_processed_folders)==0:
    dbutils.notebook.exit("No New Files to process")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
import re
expected_file_format=r"^PW_MW_DR_\d{8}\.csv$"
expected_columns=['DATE_OF_PRICING', 'ROW_ID', 'MODAL_PRICE']
for File_name in un_processed_folders:
    valid_files=[]
    sprcific_landing_layer_path=Landing_layer_full_path+File_name
    File_status='Pass'
    Failure_reason='None'
    for File_info in dbutils.fs.ls(sprcific_landing_layer_path):
        if not File_info.name.lower().endswith('.csv'):
            File_status='Fail'
            Failure_reason='File is not a csv File'
        elif File_info.size==0:
            File_status='Fail'
            Failure_reason='File size is zero'
        elif not re.match(expected_file_format, File_info.name):
            File_status='Fail'
            Failure_reason='The File not macted with expected format'

        else:
            try:
                test=spark.read.format("csv").option("header", True).load(File_info.path)
            except Exception as e:
                File_status='Fail'
                Failure_reason='File Not readable'
            if File_status=='Pass':
                if not set(expected_columns).issubset(test.columns):
                    File_status='Fail'
                    Failure_reason='File not contain primary column'

        if File_status=='Fail':
            dbutils.fs.mv(File_info.path, quarntine_full_path+'/'+File_name+'/'+File_info.name)
            # Send trigger
        else :
            valid_files.append(File_info.path)

        spark.sql(f"""insert into pricing_data.process_run_logs.pipeline_filevalidation_logs values(
            'pricing_data_ingestion_bronze',
            '{Folder_Name}',
            '{File_name}',
            '{File_status}',
            '{Failure_reason}',
            current_timestamp()
        )""")
    
    if len(valid_files)>0:
        df=spark.read.format("csv").option("header", True).load(valid_files)
        df=df.withColumn("lakehouse_ingestion_date", current_timestamp()).withColumn("lakehouse_updated_date", current_timestamp())

        df.write.format("delta").option("mergeSchema", True).mode('append').save(bronze_full_path)
        df.write.format("delta").option("mergeSchema", True).mode('append').saveAsTable(bronze_catalog_path)

    spark.sql(f"""insert into pricing_data.process_run_logs.pipeline_process_control values(
        'pricing_data_ingestion_bronze',
        '{File_name}',
        'Completed',
        current_timestamp()
    )""")
